# Reliability regeneration — ICC

Standalone reproduction of the inter-/intra-observer ICC tables. **Not part of the one-click pipeline.** It requires observer-remeasurement files that are not distributed; the main pipeline instead consumes the prepared `ICC_radiomics_evaluation.xlsx`.

## Setup

In [ ]:
# ==============================================================================
# Reliability regeneration — ICC (standalone)
# Requires observer-remeasurement files (NOT distributed):
#   Radiomics_v37_Final_Results.xlsx / _Intra_obs.xlsx / _Inter_obs.xlsx
#   Clinical_ICC__1.xlsx / Clinical_ICC__2.xlsx
# Produces the radiomic + clinical ICC tables used as inputs by the main pipeline.
# ==============================================================================
inst <- function(p) if(!(p %in% installed.packages()[,"Package"])) install.packages(p, repos="https://cran.rstudio.com")
for (p in c("readxl","writexl","irr","dplyr")) inst(p)
suppressPackageStartupMessages(lapply(c("readxl","writexl","irr","dplyr"), library, character.only=TRUE))
cat("\u2705 ICC environment ready\n")

## Radiomic ICC

In [ ]:
# ==============================================================================
# PI-MRONJ Radiomics ICC(2,1) Analysis Pipeline (Ultimate Failsafe Version)
# Two-way random effects, absolute agreement, single measurement + 95% CI added
# ==============================================================================

# 1. Load packages (resolves the tidyr conflict)
if(!require(readxl)) install.packages("readxl")
if(!require(dplyr)) install.packages("dplyr")
if(!require(irr)) install.packages("irr")
if(!require(stringr)) install.packages("stringr")

library(readxl)
library(dplyr)
library(irr)
library(stringr)

# 2. Load data
master <- read_excel("Radiomics_v37_Final_Results.xlsx")
intra  <- read_excel("Radiomics_v37_Final_Results_Intra_obs.xlsx")
inter  <- read_excel("Radiomics_v37_Final_Results_Inter_obs.xlsx")

# 3. Define a brute-force, fail-proof normalization function
force_clean_id <- function(x) {
  toupper(str_replace_all(as.character(x), "[^a-zA-Z0-9]", ""))
}

force_clean_date <- function(x) {
  str_replace_all(as.character(x), "[^0-9]", "")
}

# 4. Build UID (Patient + Implant + Date)
master <- master %>% mutate(UID = paste(force_clean_id(Patient_ID), force_clean_id(Implant_ID), force_clean_date(Study_Date), sep="_"))
intra  <- intra  %>% mutate(UID = paste(force_clean_id(Patient_ID), force_clean_id(Implant_ID), force_clean_date(Study_Date), sep="_"))
inter  <- inter  %>% mutate(UID = paste(force_clean_id(Patient_ID), force_clean_id(Implant_ID), force_clean_date(Study_Date), sep="_"))

# 5. Manual UID check before matching (debugging output)
cat("--- UID matching debug ---\n")
cat("Master UID Sample: ", head(master$UID, 3), "\n")
cat("Intra  UID Sample: ", head(intra$UID, 3), "\n")
cat("Inter  UID Sample: ", head(inter$UID, 3), "\n")

# 6. Extract common features
exclude_cols <- c("Patient_ID", "Implant_ID", "Study_Date", "PI-MRONJ",
                  "T0_Surgery", "Days_Since_Surgery", "Valid_Pixels", "UID")
features <- setdiff(intersect(colnames(master), colnames(intra)), exclude_cols)

cat(sprintf("\n▶ Number of radiomic features analyzed: %d\n", length(features)))

# 7. Merge (inner join)
df_intra_merged <- inner_join(master, intra, by = "UID", suffix = c("_orig", "_repeat"))
df_inter_merged <- inner_join(master, inter, by = "UID", suffix = c("_orig", "_repeat"))

cat(sprintf("▶ Intra-observer matched rows: %d\n", nrow(df_intra_merged)))
cat(sprintf("▶ Inter-observer matched rows: %d\n", nrow(df_inter_merged)))

# 8. Compute ICC and extract 95% CI
if(nrow(df_intra_merged) == 0 | nrow(df_inter_merged) == 0) {
  stop("🚨 0 matched rows. Re-check the UID construction method.")
}

# 💡 Function fix: use the base na.omit() instead of drop_na()
calc_icc_detailed <- function(df, feature_name) {
  col_orig <- paste0(feature_name, "_orig")
  col_rep  <- paste0(feature_name, "_repeat")

  # Remove rows with missing values
  sub_df <- df %>% select(all_of(col_orig), all_of(col_rep)) %>% na.omit()

  if (nrow(sub_df) < 5) {
    return(data.frame(ICC = NA, CI_lo = NA, CI_hi = NA))
  }

  res <- icc(sub_df, model = "twoway", type = "agreement", unit = "single")

  return(data.frame(
    ICC = res$value,
    CI_lo = res$lbound,
    CI_hi = res$ubound
  ))
}

# Use lapply to collect results as a list, then merge with bind_rows
# Intra-observer
intra_results <- lapply(features, function(f) {
  res <- calc_icc_detailed(df_intra_merged, f)
  res$Feature <- f
  res <- res %>% rename(ICC_Intra = ICC, CI_lo_Intra = CI_lo, CI_hi_Intra = CI_hi)
  return(res)
}) %>% bind_rows()

# Inter-observer
inter_results <- lapply(features, function(f) {
  res <- calc_icc_detailed(df_inter_merged, f)
  res$Feature <- f
  res <- res %>% rename(ICC_Inter = ICC, CI_lo_Inter = CI_lo, CI_hi_Inter = CI_hi)
  return(res)
}) %>% bind_rows()

# Merge intra and inter results
icc_results <- full_join(intra_results, inter_results, by = "Feature")

# 9. Round decimals and add a manuscript-ready string column
icc_results <- icc_results %>%
  select(Feature, ICC_Intra, CI_lo_Intra, CI_hi_Intra, ICC_Inter, CI_lo_Inter, CI_hi_Inter) %>%
  mutate(across(where(is.numeric), ~round(., 3))) %>%
  # Build an 'ICC (lower - upper)' string column ready to paste into the manuscript
  mutate(
    Intra_95CI_String = ifelse(is.na(ICC_Intra), NA, paste0(ICC_Intra, " (", CI_lo_Intra, "-", CI_hi_Intra, ")")),
    Inter_95CI_String = ifelse(is.na(ICC_Inter), NA, paste0(ICC_Inter, " (", CI_lo_Inter, "-", CI_hi_Inter, ")"))
  )

# Filtering logic (works on the numeric ICC_Intra / ICC_Inter columns)
stable_features <- icc_results %>% filter(ICC_Intra >= 0.75 & ICC_Inter >= 0.75)
excellent_features <- icc_results %>% filter(ICC_Intra >= 0.90 & ICC_Inter >= 0.90)

# Print and save results
cat("\n=== Final ICC summary (Intra / Inter) ===\n")
print(summary(icc_results[, c("ICC_Intra", "ICC_Inter")]))

# fileEncoding = "UTF-8" (or CP949) is recommended to avoid character corruption
write.csv(icc_results, "ICC_all_features_with_CI.csv", row.names = FALSE, fileEncoding = "UTF-8")
write.csv(stable_features, "ICC_stable_features_with_CI.csv", row.names = FALSE, fileEncoding = "UTF-8")

cat("\n✅ Analysis complete. CSV with 95% CI saved successfully.\n")

## Assemble ICC_radiomics_evaluation.xlsx (pipeline input)

In [ ]:
# ── Build ICC_radiomics_evaluation.xlsx (pipeline input) ─────────────────
#    Combines the radiomic ICC results into the 2-sheet workbook the main pipeline
#    (Multi-GEE) consumes. Keeps only Feature / ICC_Intra / ICC_Inter; the reproducibility
#    gate is ICC_Inter >= 0.75, applied on sheet 'ICC_stable_features'.
#    (This replaces the manual CSV→xlsx combination step.)
library(writexl); library(dplyr)

out_path <- "/content/ICC_radiomics_evaluation.xlsx"   # where the one-click pipeline reads it
write_xlsx(
  list(
    ICC_all_features    = icc_results    %>% select(Feature, ICC_Intra, ICC_Inter),
    ICC_stable_features = stable_features %>% select(Feature, ICC_Intra, ICC_Inter)
  ),
  path = out_path
)
cat(sprintf("\u2705 %s written  (all: %d / stable: %d features)\n",
            out_path, nrow(icc_results), nrow(stable_features)))
cat("   \u2192 download it from the Colab Files pane (left sidebar) if needed.\n")

## Clinical ICC

In [ ]:
# ==============================================================================
# PI-MRONJ Clinical Feature ICC(2,1) Analysis (Simplified for 2 Datasets)
# Two-way random effects, absolute agreement, single measurement
# ==============================================================================

# 1. Install and load packages (Colab environment)
if(!require(dplyr))   install.packages("dplyr")
if(!require(irr))     install.packages("irr")
if(!require(writexl)) install.packages("writexl")
if(!require(readxl))  install.packages("readxl")

library(dplyr)
library(irr)
library(writexl)
library(readxl)

# ==============================================================================
# 2. Load data
# Uncomment the block matching the file extension uploaded to Colab.
# ==============================================================================
# If the files are CSV:
#d1 <- read.csv("#1.xlsx - #1 icc.csv", stringsAsFactors = FALSE) # Master
#d2 <- read.csv("#2.xlsx - #2 icc.csv", stringsAsFactors = FALSE) # 2nd Observer

# If the files are Excel (.xlsx) (comment out the CSV block above and use this):
d1 <- read_excel("Clinical_ICC__1.xlsx")
d2 <- read_excel("Clinical_ICC__2.xlsx")

# ==============================================================================
# 3. Define the clinical variables to analyze (matching the actual uploaded column names)
# ==============================================================================
clin_vars <- c(
  "Insertion_Depth_mm",    # or "Calculated_insertion_depth"
  "Insertion_Angle_deg",
  "Crown_Height_mm",
  "Calculated_C_I_Ratio"
)

# ==============================================================================
# 4. Build UID from Patient_ID + Implant_ID and match
# ==============================================================================
make_uid <- function(df) {
  df %>%
    mutate(UID = paste(
      toupper(gsub("[^a-zA-Z0-9]", "", as.character(Patient_ID))),
      toupper(gsub("[^a-zA-Z0-9]", "", as.character(Implant_ID))),
      sep = "_"
    )) %>%
    group_by(UID) %>%
    slice(1) %>% # keep only the first row per implant
    ungroup()
}

d1_uid <- make_uid(d1)
d2_uid <- make_uid(d2)

cat("=== Dataset sizes ===\n")
cat(sprintf("Dataset 1: %d implants\n", nrow(d1_uid)))
cat(sprintf("Dataset 2: %d implants\n", nrow(d2_uid)))

# Inner-join the two datasets on UID
df_matched <- inner_join(
  d1_uid %>% select(UID, any_of(clin_vars)),
  d2_uid %>% select(UID, any_of(clin_vars)),
  by = "UID", suffix = c("_1", "_2")
)

cat(sprintf("\n=== Matching results ===\n"))
cat(sprintf("Total matched implants: %d pairs\n", nrow(df_matched)))

if(nrow(df_matched) == 0) {
  stop("No matched data. Check the UID construction rule or the Patient/Implant ID format.")
}

# ==============================================================================
# 5. Define the ICC(2,1) calculation function
# ==============================================================================
calc_icc <- function(df, var) {
  col_1 <- paste0(var, "_1")
  col_2 <- paste0(var, "_2")

  # Check that the variable exists in both datasets
  if (!col_1 %in% names(df) | !col_2 %in% names(df)) {
    return(c(ICC=NA, CI_lo=NA, CI_hi=NA, N=0))
  }

  # Keep only rows with no missing values (NA)
  sub <- df %>% select(all_of(c(col_1, col_2))) %>% drop_na()

  if (nrow(sub) < 3) {
    return(c(ICC=NA, CI_lo=NA, CI_hi=NA, N=nrow(sub)))
  }

  # Two-way random, Absolute agreement, Single measurement
  res <- icc(sub, model="twoway", type="agreement", unit="single")

  c(ICC   = round(res$value, 3),
    CI_lo = round(res$lbound, 3),
    CI_hi = round(res$ubound, 3),
    N     = nrow(sub))
}

# ==============================================================================
# 6. Aggregate and grade the results
# ==============================================================================
icc_table <- data.frame(Variable = clin_vars, stringsAsFactors = FALSE)

for (v in clin_vars) {
  res <- calc_icc(df_matched, v)

  icc_table$ICC[icc_table$Variable == v]   <- res["ICC"]
  icc_table$`95% CI`[icc_table$Variable == v] <- sprintf("%.3f – %.3f", res["CI_lo"], res["CI_hi"])
  icc_table$N[icc_table$Variable == v]     <- res["N"]
}

# Assign reliability grade per Koo & Li (2016) criteria
grade <- function(icc) {
  ifelse(is.na(icc), "—",
  ifelse(icc >= 0.90, "Excellent",
  ifelse(icc >= 0.75, "Good",
  ifelse(icc >= 0.50, "Moderate", "Poor"))))
}

icc_table$Grade <- grade(icc_table$ICC)

# ==============================================================================
# 7. Print results
# ==============================================================================
cat("\n=== Clinical Feature ICC(2,1) Results ===\n")
cat(sprintf("%-30s %6s %-18s %-12s %s\n", "Variable", "N", "ICC", "95% CI", "Grade"))
cat(strrep("-", 80), "\n")
for (i in seq_len(nrow(icc_table))) {
  r <- icc_table[i, ]
  cat(sprintf("%-30s %6d %6.3f   %-16s %s\n",
              r$Variable, r$N, r$ICC, r$`95% CI`, r$Grade))
}

# ==============================================================================
# 8. Save to Excel file
# ==============================================================================
write_xlsx(icc_table, "Final_ICC_Results.xlsx")
cat("\n✅ Analysis complete and saved: Final_ICC_Results.xlsx\n")